### Demo - Responses API, Chat Completeions API, Gradio UI

In [1]:
# Do the necessary imports at the top of the file(Recommended not mandatory)
# Import the dotenv library
from dotenv import load_dotenv
# import os to access the environment variables
import os
# import the OpenAI library 
from openai import OpenAI

# Print in the nice format
from IPython.display import Markdown, display

In [2]:
# it's time to load the API keys into environment variables
load_dotenv()

True

In [3]:

# Read the API key from the environment variable
openai_api_key = os.getenv('OPENAI_API_KEY')
# Check if the API key exists and print the first 6 characters
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:6]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")

OpenAI API Key exists and begins sk-pro


### Method - 1 Calling an LLM Using the OpenAI Responses API

- Creates a client object that allows us to send requests to OpenAI models.
- The client uses your environment variable OPENAI_API_KEY automatically.
- Invoke client.responses.create() by passing the necessary arguments. If you do not pass any model, it will take the default model at that time used.

#### Input Type
- Input is a string.
- The string is the prompt you send to the model.

#### Output Type
- An array of content generated by the model is in the output property of the response. 
- response.output_text is a string.

- Reference: https://platform.openai.com/docs/guides/text

In [4]:
# Create an instance of the OpenAI class
client = OpenAI()

# Create a Response from the OpenAI API - Result is a Response object
response = client.responses.create(
    model="gpt-5-nano",
    input="Write a one-sentence bedtime story about a unicorn."
)
# Print the output text of str from the Response object
#print(response)
print(response.output_text)

Under the pale moonlight, a gentle unicorn tiptoed to a mossy glade, curled beside a sighing brook, and sang a soft lullaby that carried all the sleeping creatures into gentle dreams.


In [3]:
# Print in the nice format
from IPython.display import Markdown, display
display(Markdown(response.output_text))

Under a moonlit sky, a gentle unicorn tucked his silver horn into a pillow of twinkling clouds and drifted off to dreamland.

### Method - 2 Chat Completions API


- Creates a client object that allows us to send requests to OpenAI models.
- The client uses your environment variable OPENAI_API_KEY automatically.
- Invoke client.chat.completions.create() by passing the necessary arguments. If you do not pass any model, it will take the default model at that time used. 
- Chat completions takes messages as a list of dictionaries.
- Chat Completions can return multiple parallel generations as choices, using the n param

In [6]:
# messages = "What is the capital of USA?" # Not a valid argument for the OpenAI API
messages = [
 {"role": "user", "content": "What is the capital of USA?"}   
]

In [7]:
# create an instance of the OpenAI class        
client = OpenAI()
#Get response from OpenAI API
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)
# Print the response object
print(response)

# Print the response content
print(response.choices[0].message.content)



ChatCompletion(id='chatcmpl-DMGOGgZ4rmn7Axy4rblxmBUtkK5cX', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of the United States is Washington, D.C.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774198592, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_469ff8d704', usage=CompletionUsage(completion_tokens=12, prompt_tokens=14, total_tokens=26, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
The capital of the United States is Washington, D.C.


#### Model Roles, Instructions and Reasoning

In [8]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5-mini",
    # How much thinking effort the model should use
    reasoning={"effort": "low"},
     # Style or behavior
    instructions="Talk like a pirate.",
    # User question
    input="Are semicolons optional in JavaScript?",
    
)

print(response.output_text)


Arrr! Short answer: not always — JavaScript be doin' Automatic Semicolon Insertion (ASI), so many times ye can omit semicolons, but there be gotchas where omittin' them will make yer code walk the plank.

What ASI does
- If a line break comes where the parser expects a semicolon, JS will often insert one for ye. That be why code like:
  let x = 5
  let y = x + 1
  works fine.

Common situations that break without semicolons
- return followed by a new line:
  function f() {
    return
    { a: 1 }
  }
  This returns undefined, not the object. The parser inserts a semicolon after return.
- Lines starting with (, [, `, +, -, / can be treated as continuations of the previous expression:
  let a = b
  (c).doSomething()
  This can throw or do weird things because the '(' might be parsed as a function call on b.
- ++ and -- when placed on the next line:
  x
  ++
  y
  will not do what ye expect.
- When you rely on automatic semicolons to separate statements in complex expressions, ASI can mis

In [9]:
# Language translator Assistant with Dynammic input 
def language_translation(language, text):
    client = OpenAI()

    response = client.responses.create(
        model="gpt-5-mini",
        instructions="You are an expert in Language Translation",
        input=f"Translate the given english {text} input into requested {language}",   
    )
    print(response.output_text)

In [6]:
# Calling a Function
language_translation("Tamil","Renuka Mohanraj")

ரேணுகா மோகன்ராஜ்


In [10]:

system_prompt = "You are a helpful assistant that can answer questions asked by the user in friendly manner with precise response"

#### Creating Chat Interface using Completions API and Gradio UI

In [11]:
# Defining System prompt message
system_prompt = "You are a helpful assistant that can answer questions asked by the user in friendly manner with precise response"
# Define a chat function, there are two roles in the chat: system and user
def chat(message, history):
    client = OpenAI()
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [12]:
# Creating a simple chatbot user interface using gradio
import gradio as gr     # Creates a web interface for the chatbot
gr.ChatInterface(chat, 
title="Conversation with AI",
type="messages").launch(share=True, inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://34113ac639cf5eff50.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
